In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path
from conexion_host import ConexionHost
from JUEGO_BODEGA import juego_bodega
from simondice import ejecutar_minijuego_simon

BASE_DIR = Path(__file__).resolve().parent
CONFIG_PATH = BASE_DIR / "config_grupo.json"

# Cambiar este valor para solicitar otra etapa al host de pruebas.
ETAPA_PRUEBA = "R1"

pendientes = []


def esperar_mensaje(conexion, condicion, timeout=None):
    inicio = time.time()
    while True:
        pendientes.extend(conexion.leer_nuevos_json())

        for msg in pendientes:
            if condicion(msg):
                pendientes.remove(msg)
                return msg

        if timeout is not None and time.time() - inicio > timeout:
            return None

        time.sleep(0.3)

def cargar_config():
    return json.loads(CONFIG_PATH.read_text(encoding="utf-8"))


def probar_ronda_1():
    config = cargar_config()
    conexion = ConexionHost(config)
    player_id = config.get("player_id", "P01")

    try:
        print("Conectando al host de pruebas...")
        conexion.conectar()

        # 1. Join -> Accepted
        event_id_join = f"{player_id}-00001"
        conexion.enviar_json({
            "timestamp": datetime.now().isoformat(),
            "player_id": player_id,
            "Action": "Join",
            "EventID": event_id_join,
            "Stage": "Lobby"
        })
        esperar_mensaje(
            conexion,
            lambda m: m.get("Action") == "Accepted" and m.get("ReplyTo") == event_id_join
        )
        print("¡Ingreso al Lobby confirmado!")

        # 2. StageStart (si ya llegó antes, queda en la cola y se encuentra solo)
        esperar_mensaje(
            conexion,
            lambda m: m.get("Action") == "StageStart" and m.get("Stage") == ETAPA_PRUEBA
        )
        print("¡StageStart recibido!")

        # 3. StageReady -> StageActive
        conexion.enviar_json({
            "timestamp": datetime.now().isoformat(),
            "player_id": player_id,
            "Action": "StageReady",
            "EventID": f"{player_id}-ready-r1",
            "Stage": ETAPA_PRUEBA
        })
        esperar_mensaje(
            conexion,
            lambda m: m.get("Action") == "StageActive" and m.get("Stage") == ETAPA_PRUEBA
        )
        print(f"¡La etapa {ETAPA_PRUEBA} está ACTIVA!")

        # 4. Ciclo de minijuegos: un mensaje a la vez
        intento_contador = 0
        while True:
            msg = esperar_mensaje(
                conexion,
                lambda m: m.get("Action") in ("Assign", "StageStart", "FinalStart", "GameOver")
            )
            accion = msg.get("Action")

            if accion != "Assign":
                print(f"Cambio de etapa o fin de partida ({accion}). Saliendo.")
                break

            game_id = msg.get("GameID")
            attempt = msg.get("Attempt")
            print(f"\n[ASIGNACIÓN] Minijuego ID: {game_id}, Intento: {attempt}")

            if game_id == 1:
                respuesta = juego_bodega()
            elif game_id == 2:
                respuesta = ejecutar_minijuego_simon(max_time=15)
            else:
                print(f"GameID {game_id} desconocido.")
                respuesta = {"Result": "Lose", "Score": 0}

            intento_contador += 1
            event_id_res = f"{player_id}-res-{intento_contador}"
            conexion.enviar_json({
                "timestamp": datetime.now().isoformat(),
                "player_id": player_id,
                "Action": "Result",
                "EventID": event_id_res,
                "Stage": ETAPA_PRUEBA,
                "GameID": game_id,
                "Attempt": attempt,
                "Result": respuesta["Result"],
                "Score": respuesta["Score"]
            })
            print(f"Result enviado (EventID: {event_id_res})")

            esperar_mensaje(
                conexion,
                lambda m: m.get("Action") == "ResultAccepted" and m.get("ReplyTo") == event_id_res
            )
            print("¡ResultAccepted recibido!")

    except Exception as e:
        print(f"Error durante la prueba de R1: {e}")
    finally:
        conexion.cerrar_conexion()
        print("Conexión cerrada de forma ordenada.")

if __name__ == "__main__":
    probar_ronda_1()
